In [11]:
import pandas as pd
from sklearn.metrics import classification_report, accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import time
import os
from imblearn.over_sampling import SMOTE

In [12]:
# Constants
NUM_CLASSES = 4
RANDOM_STATE = 1082
TRAIN_SIZE = 0.2
DATA_PATH = "../../../Datasets/excel/2dSpoofingDataset/GPS_Data_Simplified_2D_Feature_Map.xlsx"
MODEL_NAME = "LSTM Classifier (SMOTE)"
DATASET_NAME = "GPS_Data_Simplified_2D_Feature_Map.xlsx"
RESULTS_CSV = "../../../2D_Results.csv"

In [13]:
# Hyperparameters
INPUT_SIZE = 1
BATCH_SIZE = 64
HIDDEN_SIZE = 64
NUM_EPOCHS = 100
LEARNING_RATE = 0.001

In [14]:
# Load the dataset
try:
    df = pd.read_excel(DATA_PATH)
except FileNotFoundError:
    print("Error: The file '../Datasets/excel/GPS_Data_Simplified_2D_Feature_Map.xlsx' was not found.")
    print("Please make sure the path is correct.")
    exit()

In [15]:
# Preprocessing
X = df.drop('Output', axis=1)
y = df['Output']

print(f"\nFeatures (X) shape: {X.shape}")
print(f"Target (y) shape: {y.shape}")

print("\nSplitting the dataset into training and testing sets...")
X_train, X_test, y_train, y_test = train_test_split(
    X, y, train_size=TRAIN_SIZE, random_state=RANDOM_STATE, stratify=y
)

print(f"Training features shape: {X_train.shape}")
print(f"Testing features shape: {X_test.shape}")
print(f"Training target shape: {y_train.shape}")
print(f"Testing target shape: {y_test.shape}")

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

print("\nFeature scaling complete.")
print(f"Scaled training features shape: {X_train_scaled.shape}")
print(f"Scaled testing features shape: {X_test_scaled.shape}")

majority_class = y_train.value_counts().idxmax()
count_of_majority_class = y_train.value_counts().max()

desired_strategy = {
    1: count_of_majority_class,
    2: count_of_majority_class,
    3: count_of_majority_class
}
smote = SMOTE(sampling_strategy=desired_strategy, random_state=RANDOM_STATE)
X_train_res, y_train_res = smote.fit_resample(X_train_scaled, y_train)

# prepare data for LSTM
X_train_np = X_train_res.astype('float32').values
X_test_np = X_test_scaled.values.astype('float32')
y_train_np = y_train_res.astype('int64')
y_test_np = y_test.values.astype('int64')
SEQUENCE_LENGTH = X_train_np.shape[1]

X_train_lstm = X_train_np.reshape(-1, SEQUENCE_LENGTH, INPUT_SIZE)
X_test_lstm = X_test_np.reshape(-1, SEQUENCE_LENGTH, INPUT_SIZE)


Features (X) shape: (510530, 13)
Target (y) shape: (510530,)

Splitting the dataset into training and testing sets...
Training features shape: (102106, 13)
Testing features shape: (408424, 13)
Training target shape: (102106,)
Testing target shape: (408424,)

Feature scaling complete.
Scaled training features shape: (102106, 13)
Scaled testing features shape: (408424, 13)


In [16]:
# Convert to torch tensors
X_train_tensor = torch.tensor(X_train_lstm)
y_train_tensor = torch.tensor(y_train_np)
X_test_tensor = torch.tensor(X_test_lstm)
y_test_tensor = torch.tensor(y_test_np)

In [17]:
# Create DataLoader for batching
train_data = TensorDataset(X_train_tensor, y_train_tensor)
test_data = TensorDataset(X_test_tensor, y_test_tensor)
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE)

In [18]:
# LSTM
class LSTMClassifier(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super(LSTMClassifier, self).__init__()
        self.lstm = nn.LSTM(input_size=input_size, hidden_size=hidden_size, batch_first=True, num_layers=2)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = out[:, -1, :]
        out = self.fc(out)
        return out

In [19]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Set torch random seed
torch.manual_seed(RANDOM_STATE)
if torch.cuda.is_available():
    torch.cuda.manual_seed(RANDOM_STATE)
    torch.cuda.manual_seed_all(RANDOM_STATE)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [20]:
# initialize the model, loss function, and optimizer
model = LSTMClassifier(INPUT_SIZE, HIDDEN_SIZE, NUM_CLASSES).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

In [21]:
# Training loop
start_time = time.time()

for epoch in range(NUM_EPOCHS):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * X_batch.size(0)

        # Calculate accuracy
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == y_batch).sum().item()
        total += y_batch.size(0)

    epoch_loss = running_loss / len(train_loader.dataset)
    epoch_acc = correct / total
    print(f"Epoch {epoch+1}/{NUM_EPOCHS}, Loss: {epoch_loss:.4f}, Accuracy: {epoch_acc:.4f}")

train_time = time.time() - start_time

Epoch 1/100, Loss: 0.5219, Accuracy: 0.7745
Epoch 2/100, Loss: 0.3301, Accuracy: 0.8606
Epoch 3/100, Loss: 0.2862, Accuracy: 0.8799
Epoch 4/100, Loss: 0.2538, Accuracy: 0.8941
Epoch 5/100, Loss: 0.2262, Accuracy: 0.9053
Epoch 6/100, Loss: 0.2115, Accuracy: 0.9110
Epoch 7/100, Loss: 0.1982, Accuracy: 0.9166
Epoch 8/100, Loss: 0.1901, Accuracy: 0.9193
Epoch 9/100, Loss: 0.1846, Accuracy: 0.9214
Epoch 10/100, Loss: 0.1790, Accuracy: 0.9236
Epoch 11/100, Loss: 0.1755, Accuracy: 0.9251
Epoch 12/100, Loss: 0.1694, Accuracy: 0.9273
Epoch 13/100, Loss: 0.1644, Accuracy: 0.9296
Epoch 14/100, Loss: 0.1636, Accuracy: 0.9303
Epoch 15/100, Loss: 0.1582, Accuracy: 0.9322
Epoch 16/100, Loss: 0.1576, Accuracy: 0.9329
Epoch 17/100, Loss: 0.1535, Accuracy: 0.9348
Epoch 18/100, Loss: 0.1504, Accuracy: 0.9362
Epoch 19/100, Loss: 0.1500, Accuracy: 0.9360
Epoch 20/100, Loss: 0.1451, Accuracy: 0.9385
Epoch 21/100, Loss: 0.1437, Accuracy: 0.9392
Epoch 22/100, Loss: 0.1409, Accuracy: 0.9409
Epoch 23/100, Loss:

In [22]:
# Evaluation
model.eval()
all_preds = []
all_labels = []
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device)
        outputs = model(X_batch)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y_batch.numpy())

print("\nClassification Report:")
print(classification_report(all_labels, all_preds, digits=4))
accuracy = accuracy_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds, average='weighted', zero_division=0)
print("Accuracy:", accuracy * 100)
print("F1 Score:", f1 * 100)


Classification Report:
              precision    recall  f1-score   support

           0     0.9846    0.9337    0.9584    318260
           1     0.7272    0.8449    0.7816     29166
           2     0.6546    0.8421    0.7366     35386
           3     0.8991    0.9549    0.9261     25612

    accuracy                         0.9207    408424
   macro avg     0.8163    0.8939    0.8507    408424
weighted avg     0.9322    0.9207    0.9246    408424

Accuracy: 92.07294380349833
F1 Score: 92.45683099368206


In [23]:
# Save results to CSV
results = [{
    "model": MODEL_NAME,
    "dataset": DATASET_NAME,
    "f1score": f1,
    "accuracy": accuracy,
    "train_time_sec": train_time,
    "parameters": str(model.state_dict().keys()),
    "num_epochs": NUM_EPOCHS
}]
results_df = pd.DataFrame(results)

file_exists = os.path.isfile(RESULTS_CSV)
results_df.to_csv(
    RESULTS_CSV,
    mode='a',
    header=not file_exists,
    index=False
)
print(f"Results appended to {RESULTS_CSV}")

Results appended to ../../../2D_Results.csv
